In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

analysis_master = spark.table(
    "workspace.default.applebees_analysis_master"
)

print("Rows:", analysis_master.count())

Rows: 2142846


Carga de datos

In [0]:
# ==========================================
# FRAUD EDA - 1. Discount Overview
# ==========================================

discount_overview = analysis_master.select(
    F.count("*").alias("total_checks"),

    F.sum(
        F.when(F.col("discount_count") > 0, 1).otherwise(0)
    ).alias("checks_with_discount"),

    F.round(
        F.avg("discount_amount"),
        2
    ).alias("avg_discount_all_checks"),

    F.round(
        F.avg(
            F.when(
                F.col("discount_count") > 0,
                F.col("discount_amount")
            )
        ),
        2
    ).alias("avg_discount_discounted_checks"),

    F.round(
        F.max("discount_amount"),
        2
    ).alias("max_total_discount")
)

display(discount_overview)

total_checks,checks_with_discount,avg_discount_all_checks,avg_discount_discounted_checks,max_total_discount
2142846,214837,0.91,9.09,421.55


In [0]:
# ==========================================
# FRAUD EDA - 2. Load Event Tables
# ==========================================

discount_events = spark.table(
    "workspace.default.applebees_discount_events"
)

payment_events = spark.table(
    "workspace.default.applebees_payment_events"
)

void_events = spark.table(
    "workspace.default.applebees_void_events"
)

print("Discounts:", discount_events.count())
print("Payments:", payment_events.count())
print("Voids:", void_events.count())

Discounts: 240699
Payments: 2232092
Voids: 178303


In [0]:
# ==========================================
# FRAUD EDA - 2.2 Discount Profile by Employee
# ==========================================

employee_discount_profile = (
    discount_events
    .filter(F.col("EmployeeId") != 0)
    .groupBy("EmployeeId")
    .agg(
        F.count("*").alias("discount_events"),
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("discounted_checks"),
        F.round(F.sum("Amount"), 2).alias("total_discount_amount"),
        F.round(F.avg("Amount"), 2).alias("avg_discount_amount"),
        F.round(F.max("Amount"), 2).alias("max_discount_amount")
    )
)

print("Employees with discounts:", employee_discount_profile.count())

display(
    employee_discount_profile
    .orderBy(F.desc("total_discount_amount"))
    .limit(15)
)

Employees with discounts: 3023


EmployeeId,discount_events,discounted_checks,total_discount_amount,avg_discount_amount,max_discount_amount
13278,578,491,6371.63,11.02,59.98
28658,691,610,5784.46,8.37,52.98
13306,701,573,5600.34,7.99,50.0
42242,975,864,5060.05,5.19,91.96
110343,657,585,5002.69,7.61,99.42
113408,509,483,5001.07,9.83,44.94
22593,957,840,4831.39,5.05,55.98
135756,436,370,4714.99,10.81,51.98
14488,466,395,4548.68,9.76,50.0
82160,505,450,4400.25,8.71,36.98


In [0]:
# ==========================================
# FRAUD EDA - 2.3 Employee Activity
# ==========================================

employee_activity = (
    payment_events
    .filter(F.col("EmployeeId") != 0)
    .groupBy("EmployeeId")
    .agg(
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("total_checks")
    )
)

employee_discount_risk = (
    employee_activity
    .join(
        employee_discount_profile,
        on="EmployeeId",
        how="left"
    )
    .fillna(
        0,
        subset=[
            "discount_events",
            "discounted_checks",
            "total_discount_amount"
        ]
    )
    .withColumn(
        "discount_rate_pct",
        F.round(
            F.col("discounted_checks") /
            F.col("total_checks") * 100,
            2
        )
    )
)

display(
    employee_discount_risk
    .filter(F.col("total_checks") >= 100)
    .orderBy(F.desc("discount_rate_pct"))
    .limit(15)
)

EmployeeId,total_checks,discount_events,discounted_checks,total_discount_amount,avg_discount_amount,max_discount_amount,discount_rate_pct
42242,1896,975,864,5060.05,5.19,91.96,45.57
52491,593,292,263,797.54,2.73,23.99,44.35
111465,939,386,359,2029.22,5.26,30.0,38.23
5860,1548,673,584,1982.12,2.95,24.25,37.73
114426,414,147,137,697.17,4.74,30.0,33.09
22593,2557,957,840,4831.39,5.05,55.98,32.85
20383,909,367,284,3609.83,9.84,32.98,31.24
150857,296,102,92,1478.96,14.5,30.0,31.08
104918,257,111,79,1198.14,10.79,33.5,30.74
125811,970,314,295,1219.18,3.88,26.0,30.41


_Al normalizar los descuentos por el volumen de actividad de cada empleado, se identificaron diferencias importantes en su comportamiento. Mientras que aproximadamente 10% de los checks del dataset presentan descuentos, algunos empleados registran descuentos en más del 40% de sus checks asociados. Además, se observaron perfiles distintos entre empleados con descuentos frecuentes de bajo monto y empleados con menor frecuencia pero montos promedio mayores. Estos resultados no demuestran fraude, pero permiten identificar comportamientos atípicos que requieren un análisis conjunto con otras señales, como voids y gratuity._

Voids

In [0]:
# ==========================================
# FRAUD EDA - 3.1 Void Rate by Employee
# ==========================================

employee_void_profile = (
    void_events
    .filter(F.col("EmployeeId") != 0)
    .groupBy("EmployeeId")
    .agg(
        F.count("*").alias("void_events"),
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("voided_checks"),
        F.round(F.sum("Amount"), 2).alias("total_void_amount"),
        F.round(F.avg("Amount"), 2).alias("avg_void_amount"),
        F.round(F.max("Amount"), 2).alias("max_void_amount")
    )
)

employee_void_risk = (
    employee_activity
    .join(
        employee_void_profile,
        on="EmployeeId",
        how="left"
    )
    .fillna(
        0,
        subset=[
            "void_events",
            "voided_checks",
            "total_void_amount"
        ]
    )
    .withColumn(
        "void_rate_pct",
        F.round(
            F.col("voided_checks") /
            F.col("total_checks") * 100,
            2
        )
    )
)

display(
    employee_void_risk
    .filter(F.col("total_checks") >= 100)
    .orderBy(F.desc("void_rate_pct"))
    .limit(15)
)

EmployeeId,total_checks,void_events,voided_checks,total_void_amount,avg_void_amount,max_void_amount,void_rate_pct
161078,247,205,43,762.17,3.72,21.49,17.41
159512,376,247,60,868.02,3.51,25.99,15.96
161653,202,99,31,482.06,4.87,25.0,15.35
160674,146,130,21,567.15,4.36,22.99,14.38
159062,168,82,24,388.62,4.74,23.53,14.29
160171,184,121,26,528.43,4.37,20.99,14.13
161205,263,176,37,633.1,3.6,26.99,14.07
161717,115,27,16,139.06,5.15,15.49,13.91
160245,260,126,36,591.37,4.69,22.99,13.85
160026,416,250,53,967.47,3.87,18.99,12.74


#### Insights principales

Se identificaron diferencias importantes en la frecuencia de voids entre empleados. Entre aquellos con al menos 100 checks asociados, las tasas más altas alcanzan aproximadamente el 17%. Además, algunos empleados presentan una cantidad considerablemente mayor de eventos de void que de checks con void, lo que indica concentración de múltiples voids dentro de una misma transacción.

Estos resultados no representan evidencia de fraude por sí solos, pero permiten identificar empleados y patrones que requieren mayor investigación considerando conjuntamente frecuencia, repetición y monto de los voids.

In [0]:
# ==========================================
# FRAUD EDA - 3.2 Repeated Voids by Employee
# ==========================================

repeated_void_checks = (
    void_events
    .filter(F.col("EmployeeId") != 0)
    .groupBy(
        "EmployeeId",
        "LocationId",
        "BusinessDate",
        "CheckId"
    )
    .agg(
        F.count("*").alias("void_events_in_check"),
        F.round(F.sum("Amount"), 2).alias("void_amount_in_check")
    )
    .filter(F.col("void_events_in_check") > 1)
)

employee_repeated_voids = (
    repeated_void_checks
    .groupBy("EmployeeId")
    .agg(
        F.count("*").alias("checks_with_multiple_voids"),
        F.sum("void_events_in_check").alias("void_events_in_repeated_checks"),
        F.round(
            F.avg("void_events_in_check"), 2
        ).alias("avg_voids_per_repeated_check"),
        F.max("void_events_in_check").alias("max_voids_in_single_check")
    )
)

employee_repeated_void_risk = (
    employee_void_risk
    .join(
        employee_repeated_voids,
        on="EmployeeId",
        how="left"
    )
    .fillna(
        0,
        subset=[
            "checks_with_multiple_voids",
            "void_events_in_repeated_checks",
            "avg_voids_per_repeated_check",
            "max_voids_in_single_check"
        ]
    )
    .withColumn(
        "repeated_void_rate_pct",
        F.round(
            F.col("checks_with_multiple_voids") /
            F.col("total_checks") * 100,
            2
        )
    )
)

display(
    employee_repeated_void_risk
    .filter(F.col("total_checks") >= 100)
    .orderBy(F.desc("repeated_void_rate_pct"))
    .limit(15)
)

EmployeeId,total_checks,void_events,voided_checks,total_void_amount,avg_void_amount,max_void_amount,void_rate_pct,checks_with_multiple_voids,void_events_in_repeated_checks,avg_voids_per_repeated_check,max_voids_in_single_check,repeated_void_rate_pct
161078,247,205,43,762.17,3.72,21.49,17.41,35,197,5.63,29,14.17
160674,146,130,21,567.15,4.36,22.99,14.38,18,127,7.06,23,12.33
161653,202,99,31,482.06,4.87,25.0,15.35,23,91,3.96,9,11.39
160171,184,121,26,528.43,4.37,20.99,14.13,20,115,5.75,20,10.87
161205,263,176,37,633.1,3.6,26.99,14.07,26,165,6.35,31,9.89
159512,376,247,60,868.02,3.51,25.99,15.96,37,224,6.05,72,9.84
160026,416,250,53,967.47,3.87,18.99,12.74,40,237,5.93,17,9.62
159502,260,167,30,596.21,3.57,18.99,11.54,24,161,6.71,38,9.23
160575,255,128,28,541.37,4.23,25.0,10.98,23,123,5.35,32,9.02
160541,111,63,13,259.01,4.11,18.99,11.71,10,60,6.0,23,9.01


#### Insights principales

El análisis muestra que algunos empleados concentran múltiples eventos de void dentro de una misma transacción. El EmployeeId `161078` presenta la mayor tasa de checks con múltiples voids (14.17%), con 197 eventos concentrados en 35 checks y un promedio de 5.63 voids por check repetido.

También se identificaron casos de alta concentración dentro de transacciones individuales. Por ejemplo, un empleado registra hasta 72 eventos de void dentro de un solo check.

Estos patrones no demuestran fraude, pero representan señales relevantes para priorizar transacciones y empleados que requieren una investigación más detallada.

Patrones combinados

In [0]:
# ==========================================
# FRAUD EDA - 4.1 Discounts + Voids
# ==========================================

employee_combined = (
    employee_discount_risk
    .select(
        "EmployeeId",
        "total_checks",
        "discounted_checks",
        "discount_events",
        "total_discount_amount",
        "avg_discount_amount",
        "discount_rate_pct"
    )
    .join(
        employee_void_risk.select(
            "EmployeeId",
            "voided_checks",
            "void_events",
            "total_void_amount",
            "avg_void_amount",
            "void_rate_pct"
        ),
        on="EmployeeId",
        how="left"
    )
    .fillna(
        0,
        subset=[
            "voided_checks",
            "void_events",
            "total_void_amount",
            "void_rate_pct"
        ]
    )
)

display(
    employee_combined
    .filter(F.col("total_checks") >= 100)
    .orderBy(
        F.desc("discount_rate_pct"),
        F.desc("void_rate_pct")
    )
    .limit(20)
)

EmployeeId,total_checks,discounted_checks,discount_events,total_discount_amount,avg_discount_amount,discount_rate_pct,voided_checks,void_events,total_void_amount,avg_void_amount,void_rate_pct
42242,1896,864,975,5060.05,5.19,45.57,20,39,185.39,4.75,1.05
52491,593,263,292,797.54,2.73,44.35,10,74,151.68,2.05,1.69
111465,939,359,386,2029.22,5.26,38.23,56,319,1119.57,3.51,5.96
5860,1548,584,673,1982.12,2.95,37.73,33,153,546.7,3.57,2.13
114426,414,137,147,697.17,4.74,33.09,5,19,109.59,5.77,1.21
22593,2557,840,957,4831.39,5.05,32.85,31,81,333.01,4.11,1.21
20383,909,284,367,3609.83,9.84,31.24,30,76,277.68,3.65,3.3
150857,296,92,102,1478.96,14.5,31.08,17,51,215.49,4.23,5.74
104918,257,79,111,1198.14,10.79,30.74,26,100,384.05,3.84,10.12
125811,970,295,314,1219.18,3.88,30.41,50,109,444.1,4.07,5.15


#### Insights principales

El análisis conjunto muestra que una tasa elevada de descuentos no necesariamente coincide con una tasa elevada de voids. Por ejemplo, el EmployeeId `42242` presenta la mayor tasa de descuentos (45.57%), pero una tasa de voids de solo 1.05%.

En contraste, se identificaron empleados con múltiples señales simultáneas. El EmployeeId `104918` registra descuentos en 30.74% de sus checks asociados y voids en 10.12%, mientras que `111465` presenta tasas de 38.23% y 5.96%, respectivamente, sobre un mayor volumen de 939 checks.

Estos resultados muestran la importancia de combinar diferentes indicadores de comportamiento en lugar de utilizar una sola métrica para identificar transacciones o empleados que requieren investigación adicional.

In [0]:
# ==========================================
# FRAUD EDA - 4.2.1 Discounts vs Gratuity
# ==========================================

discount_gratuity_comparison = (
    analysis_master
    .withColumn(
        "discount_group",
        F.when(
            F.col("discount_count") > 0,
            "Con descuento"
        ).otherwise("Sin descuento")
    )
    .groupBy("discount_group")
    .agg(
        F.count("*").alias("checks"),
        F.round(F.avg("gratuity_raw"), 2).alias("avg_gratuity_raw"),
        F.round(
            F.expr("percentile_approx(gratuity_raw, 0.5)"),
            2
        ).alias("median_gratuity_raw"),
        F.round(F.min("gratuity_raw"), 2).alias("min_gratuity_raw"),
        F.round(F.max("gratuity_raw"), 2).alias("max_gratuity_raw")
    )
)

display(discount_gratuity_comparison)

discount_group,checks,avg_gratuity_raw,median_gratuity_raw,min_gratuity_raw,max_gratuity_raw
Sin descuento,1928009,-4.25,0.0,-570.43,6.48
Con descuento,214837,-3.06,0.0,-500.0,0.0


#### Insights principales

Los checks con y sin descuento presentan diferencias en el promedio de `Gratuity`: -3.06 para checks con descuento y -4.25 para checks sin descuento. Sin embargo, la mediana es 0 en ambos grupos y existen valores extremos importantes.

Debido a que la convención de signos de `Gratuity` en el sistema POS no ha sido validada, estos resultados no permiten concluir que los descuentos estén asociados con mayores propinas. La relación se mantiene como una hipótesis exploratoria que requiere un análisis adicional.

In [0]:
# ==========================================
# FRAUD EDA - 4.2.2 Discount Amount vs Gratuity
# ==========================================

discount_gratuity_corr = (
    analysis_master
    .filter(
        (F.col("discount_count") > 0) &
        F.col("gratuity_raw").isNotNull()
    )
    .select(
        F.corr(
            "discount_amount",
            "gratuity_raw"
        ).alias("correlation")
    )
)

display(discount_gratuity_corr)

correlation
0.06465490661044389


#### Insight

La correlación entre el monto del descuento y `Gratuity` fue de **0.065**, lo que indica una relación lineal prácticamente inexistente entre ambas variables.

En conjunto con la comparación anterior, los resultados no proporcionan evidencia suficiente para afirmar que descuentos mayores estén asociados con mayores propinas. Además, la convención de signos de `Gratuity` continúa siendo una limitación para su interpretación.

In [0]:
# ==========================================
# FRAUD EDA - 4.3.1 Multiple Risk Signals
# ==========================================

employee_risk_signals = (
    employee_combined
    .join(
        employee_repeated_void_risk.select(
            "EmployeeId",
            "checks_with_multiple_voids",
            "avg_voids_per_repeated_check",
            "max_voids_in_single_check",
            "repeated_void_rate_pct"
        ),
        on="EmployeeId",
        how="left"
    )
    .fillna(
        0,
        subset=[
            "checks_with_multiple_voids",
            "avg_voids_per_repeated_check",
            "max_voids_in_single_check",
            "repeated_void_rate_pct"
        ]
    )
)

display(
    employee_risk_signals
    .filter(F.col("total_checks") >= 100)
    .select(
        "EmployeeId",
        "total_checks",
        "discount_rate_pct",
        "void_rate_pct",
        "repeated_void_rate_pct",
        "avg_discount_amount",
        "total_discount_amount",
        "total_void_amount",
        "max_voids_in_single_check"
    )
    .orderBy(
        F.desc("discount_rate_pct"),
        F.desc("void_rate_pct"),
        F.desc("repeated_void_rate_pct")
    )
    .limit(20)
)

EmployeeId,total_checks,discount_rate_pct,void_rate_pct,repeated_void_rate_pct,avg_discount_amount,total_discount_amount,total_void_amount,max_voids_in_single_check
42242,1896,45.57,1.05,0.47,5.19,5060.05,185.39,6
52491,593,44.35,1.69,1.52,2.73,797.54,151.68,50
111465,939,38.23,5.96,4.47,5.26,2029.22,1119.57,20
5860,1548,37.73,2.13,1.42,2.95,1982.12,546.7,48
114426,414,33.09,1.21,1.21,4.74,697.17,109.59,9
22593,2557,32.85,1.21,0.86,5.05,4831.39,333.01,10
20383,909,31.24,3.3,2.53,9.84,3609.83,277.68,6
150857,296,31.08,5.74,3.04,14.5,1478.96,215.49,22
104918,257,30.74,10.12,6.61,10.79,1198.14,384.05,28
125811,970,30.41,5.15,2.68,3.88,1219.18,444.1,14


#### Insights principales

La combinación de indicadores permitió distinguir entre empleados que presentan una sola señal atípica y aquellos que acumulan múltiples patrones simultáneamente.

El EmployeeId `104918` destaca con tasas de 30.74% en descuentos, 10.12% en voids y 6.61% en checks con múltiples voids. Por otro lado, `111465` presenta tasas de 38.23%, 5.96% y 4.47%, respectivamente, respaldadas por un mayor volumen de 939 checks.

También se identificaron casos como `140589` y `150857` con múltiples señales relevantes, aunque con menor volumen de actividad. En contraste, `42242` presenta una tasa de descuentos particularmente elevada (45.57%), pero tasas relativamente bajas de voids, mostrando que una señal extrema aislada no necesariamente representa un patrón de riesgo combinado.

Estos resultados permiten priorizar empleados para investigación adicional sin clasificarlos directamente como fraudulentos.

In [0]:
# ==========================================
# FRAUD EDA - 4.3.2 Extreme Risk Signals
# ==========================================

# Consider only employees with sufficient activity
risk_base = (
    employee_risk_signals
    .filter(F.col("total_checks") >= 100)
)

# Calculate 95th percentile thresholds
thresholds = risk_base.select(
    F.expr(
        "percentile_approx(discount_rate_pct, 0.95)"
    ).alias("p95_discount_rate"),

    F.expr(
        "percentile_approx(void_rate_pct, 0.95)"
    ).alias("p95_void_rate"),

    F.expr(
        "percentile_approx(repeated_void_rate_pct, 0.95)"
    ).alias("p95_repeated_void_rate")
).first()

p95_discount = thresholds["p95_discount_rate"]
p95_void = thresholds["p95_void_rate"]
p95_repeated = thresholds["p95_repeated_void_rate"]

print("P95 Discount Rate:", p95_discount)
print("P95 Void Rate:", p95_void)
print("P95 Repeated Void Rate:", p95_repeated)

P95 Discount Rate: 18.49
P95 Void Rate: 7.46
P95 Repeated Void Rate: 4.95


#### Umbrales de comportamiento extremo

A partir del percentil 95 de los empleados con al menos 100 checks asociados, se establecieron los siguientes umbrales exploratorios:

- **Discount Rate:** 18.49%
- **Void Rate:** 7.46%
- **Repeated Void Rate:** 4.95%

Los empleados que superan estos valores se encuentran aproximadamente dentro del 5% superior de la distribución de cada indicador. Estos umbrales no representan evidencia de fraude, sino una herramienta para identificar comportamientos atípicos que requieren investigación adicional.

In [0]:
# ==========================================
# FRAUD EDA - 4.3.3 Multiple Extreme Signals
# ==========================================

employee_extreme_signals = (
    risk_base

    .withColumn(
        "high_discount",
        F.col("discount_rate_pct") >= p95_discount
    )

    .withColumn(
        "high_void",
        F.col("void_rate_pct") >= p95_void
    )

    .withColumn(
        "high_repeated_void",
        F.col("repeated_void_rate_pct") >= p95_repeated
    )

    .withColumn(
        "extreme_signal_count",
        F.col("high_discount").cast("int") +
        F.col("high_void").cast("int") +
        F.col("high_repeated_void").cast("int")
    )
)

display(
    employee_extreme_signals
    .filter(F.col("extreme_signal_count") >= 2)
    .select(
        "EmployeeId",
        "total_checks",
        "discount_rate_pct",
        "void_rate_pct",
        "repeated_void_rate_pct",
        "high_discount",
        "high_void",
        "high_repeated_void",
        "extreme_signal_count"
    )
    .orderBy(
        F.desc("extreme_signal_count"),
        F.desc("total_checks")
    )
)

EmployeeId,total_checks,discount_rate_pct,void_rate_pct,repeated_void_rate_pct,high_discount,high_void,high_repeated_void,extreme_signal_count
114418,924,19.37,11.8,8.23,true,true,true,3
154130,786,20.99,8.91,6.62,true,true,true,3
161101,285,19.65,8.42,5.26,true,true,true,3
104918,257,30.74,10.12,6.61,true,true,true,3
159766,188,18.62,10.11,7.98,true,true,true,3
160872,185,22.7,7.57,5.41,true,true,true,3
161089,142,19.01,8.45,7.75,true,true,true,3
140589,112,25.89,9.82,6.25,true,true,true,3
161207,106,19.81,9.43,7.55,true,true,true,3
161866,103,19.42,9.71,5.83,true,true,true,3


#### Insights principales

Se identificaron **10 empleados que superan simultáneamente el percentil 95 en las tasas de descuentos, voids y voids repetidos**. Estos casos representan los perfiles con mayor combinación de señales atípicas dentro de los empleados con al menos 100 checks asociados.

Entre ellos, `114418` y `154130` destacan por presentar las tres señales extremas sobre volúmenes relativamente altos de 924 y 786 checks, respectivamente. El EmployeeId `104918`, identificado previamente durante el análisis combinado, también supera los tres umbrales y presenta una tasa de descuentos particularmente elevada de 30.74%.

Por otro lado, empleados como `161078` presentan tasas extremadamente altas de voids y voids repetidos, pero no de descuentos. Esto muestra que existen diferentes perfiles de comportamiento atípico y que no todos los casos de interés presentan las mismas señales.

Los resultados permiten priorizar casos para investigación adicional, pero no constituyen evidencia directa de fraude.

In [0]:
# ==========================================
# FRAUD EDA - 4.3.4 Priority Shortlist
# ==========================================

priority_employees = (
    employee_extreme_signals
    .filter(F.col("extreme_signal_count") == 3)
    .select(
        "EmployeeId",
        "total_checks",
        "discount_rate_pct",
        "void_rate_pct",
        "repeated_void_rate_pct",
        "total_discount_amount",
        "total_void_amount"
    )
    .orderBy(F.desc("total_checks"))
)

display(priority_employees)

EmployeeId,total_checks,discount_rate_pct,void_rate_pct,repeated_void_rate_pct,total_discount_amount,total_void_amount
114418,924,19.37,11.8,8.23,2414.95,1291.95
154130,786,20.99,8.91,6.62,917.82,841.43
161101,285,19.65,8.42,5.26,529.69,344.05
104918,257,30.74,10.12,6.61,1198.14,384.05
159766,188,18.62,10.11,7.98,241.28,345.47
160872,185,22.7,7.57,5.41,441.68,369.01
161089,142,19.01,8.45,7.75,370.57,268.33
140589,112,25.89,9.82,6.25,229.66,128.06
161207,106,19.81,9.43,7.55,288.35,235.88
161866,103,19.42,9.71,5.83,214.99,156.62


#### Insight principal

El shortlist final identifica 10 empleados que superan simultáneamente el percentil 95 en las tasas de descuentos, voids y voids repetidos.

Los EmployeeId `114418` y `154130` destacan por presentar las tres señales sobre volúmenes relativamente altos de 924 y 786 checks. Por otro lado, `104918` presenta uno de los perfiles más intensos, con 30.74% de checks con descuento, 10.12% con voids y 6.61% con múltiples voids.

Estos empleados representan casos prioritarios para investigación adicional debido a la combinación de múltiples comportamientos atípicos, no porque exista evidencia suficiente para clasificarlos como fraude.

Visualizaciónes de patrones de riesgo

In [0]:
# ==========================================
# FRAUD EDA - 5.1 Visualization Data
# ==========================================

fraud_scatter = (
    employee_extreme_signals
    .select(
        "EmployeeId",
        "total_checks",
        "discount_rate_pct",
        "void_rate_pct",
        "repeated_void_rate_pct",
        "extreme_signal_count"
    )
)

display(fraud_scatter)

EmployeeId,total_checks,discount_rate_pct,void_rate_pct,repeated_void_rate_pct,extreme_signal_count
12252,425,9.65,4.24,3.06,0
148784,2176,6.16,0.6,0.37,0
161671,262,12.6,4.2,3.05,0
152593,845,18.22,2.13,1.07,0
147889,578,16.26,6.92,5.02,1
160772,492,14.84,3.25,2.44,0
158918,805,8.07,3.35,1.99,0
156225,1011,0.3,4.85,2.47,0
154909,1553,8.69,2.51,1.48,0
24753,785,12.74,3.31,2.68,0


Databricks visualization. Run in Databricks to view.

#### Insight

La mayoría de los empleados se concentra en niveles relativamente bajos de descuentos y voids. Sin embargo, se observa un grupo reducido con tasas elevadas en ambas dimensiones.

Los empleados identificados con tres señales extremas se concentran principalmente en esta región de mayor riesgo relativo, mientras que otros casos presentan comportamientos extremos únicamente en descuentos o voids. Esto demuestra la importancia de utilizar múltiples indicadores para priorizar casos de investigación.

In [0]:
# ==========================================
# FRAUD EDA - 5.2 Priority Employees
# ==========================================

priority_chart = (
    priority_employees
    .select(
        F.col("EmployeeId").cast("string").alias("EmployeeId"),
        "total_checks",
        "discount_rate_pct",
        "void_rate_pct",
        "repeated_void_rate_pct"
    )
    .orderBy(F.desc("total_checks"))
)

display(priority_chart)

EmployeeId,total_checks,discount_rate_pct,void_rate_pct,repeated_void_rate_pct
114418,924,19.37,11.8,8.23
154130,786,20.99,8.91,6.62
161101,285,19.65,8.42,5.26
104918,257,30.74,10.12,6.61
159766,188,18.62,10.11,7.98
160872,185,22.7,7.57,5.41
161089,142,19.01,8.45,7.75
140589,112,25.89,9.82,6.25
161207,106,19.81,9.43,7.55
161866,103,19.42,9.71,5.83


Databricks visualization. Run in Databricks to view.

#### Insight

Los empleados prioritarios presentan diferentes combinaciones de comportamientos atípicos. Mientras `104918` destaca principalmente por una tasa de descuentos de 30.74%, `114418` presenta las mayores tasas de voids (11.80%) y voids repetidos (8.23%) dentro del shortlist.

La comparación demuestra que no existe un único patrón de riesgo y respalda el uso conjunto de múltiples indicadores para priorizar casos de investigación.

In [0]:
# ==========================================
# FRAUD EDA - 5.3 Void Concentration
# ==========================================

void_concentration_chart = (
    employee_repeated_void_risk
    .filter(F.col("total_checks") >= 100)
    .select(
        F.col("EmployeeId").cast("string").alias("EmployeeId"),
        "max_voids_in_single_check",
        "avg_voids_per_repeated_check",
        "repeated_void_rate_pct"
    )
    .orderBy(F.desc("max_voids_in_single_check"))
    .limit(10)
)

display(void_concentration_chart)

EmployeeId,max_voids_in_single_check,avg_voids_per_repeated_check,repeated_void_rate_pct
153297,266,19.59,1.35
129310,225,19.85,3.51
152301,192,25.75,0.62
149201,188,13.67,1.33
159311,100,9.8,8.37
143742,94,6.07,2.29
102408,88,8.63,8.52
103865,84,11.25,1.12
149928,73,11.89,1.12
159512,72,6.05,9.84


Databricks visualization. Run in Databricks to view.

#### Insights principales

Se identificaron transacciones con una concentración excepcionalmente elevada de eventos de void. El EmployeeId `153297` registra un máximo de 266 eventos dentro de un solo check, seguido de `129310` con 225 y `152301` con 192.

Sin embargo, los máximos extremos no necesariamente corresponden a empleados con mayores tasas de repetición. Por ejemplo, `159512` presenta un máximo de 72 eventos, pero una tasa de checks con múltiples voids de 9.84%.

Estos resultados permiten distinguir entre eventos aislados de gran magnitud y patrones de repetición frecuentes. Ambos requieren investigación adicional antes de considerarse posibles irregularidades.

In [0]:
# 6.1 - Verify available DataFrames

variables = [
    "priority_employees",
    "employee_discount_risk",
    "employee_void_risk",
    "employee_repeated_void_risk"
]

for variable in variables:
    print(f"{variable}: {'Disponible' if variable in globals() else 'No disponible'}")

priority_employees: Disponible
employee_discount_risk: Disponible
employee_void_risk: Disponible
employee_repeated_void_risk: Disponible


In [0]:
# 6.1 - Inspect available employee metrics

for name in [
    "priority_employees",
    "employee_discount_risk",
    "employee_void_risk",
    "employee_repeated_void_risk"
]:
    print(f"\n{name}")
    print(globals()[name].columns)


priority_employees
['EmployeeId', 'total_checks', 'discount_rate_pct', 'void_rate_pct', 'repeated_void_rate_pct', 'total_discount_amount', 'total_void_amount']

employee_discount_risk
['EmployeeId', 'total_checks', 'discount_events', 'discounted_checks', 'total_discount_amount', 'avg_discount_amount', 'max_discount_amount', 'discount_rate_pct']

employee_void_risk
['EmployeeId', 'total_checks', 'void_events', 'voided_checks', 'total_void_amount', 'avg_void_amount', 'max_void_amount', 'void_rate_pct']

employee_repeated_void_risk
['EmployeeId', 'total_checks', 'void_events', 'voided_checks', 'total_void_amount', 'avg_void_amount', 'max_void_amount', 'void_rate_pct', 'checks_with_multiple_voids', 'void_events_in_repeated_checks', 'avg_voids_per_repeated_check', 'max_voids_in_single_check', 'repeated_void_rate_pct']


In [0]:
# 6.1 - Check source columns for Employee EDA

master = spark.table(
    "workspace.default.applebees_analysis_master"
)

print("ItemsSold disponible:", "ItemsSold" in master.columns)
print("Columnas relacionadas con ventas:")
print([c for c in master.columns if "sold" in c.lower() or "sales" in c.lower()])

ItemsSold disponible: True
Columnas relacionadas con ventas:
['ItemsSold', 'total_sold_price']


In [0]:
# 6.1 - Employee Sales from ItemsSold

employee_sales = (
    master
    .select(F.explode("ItemsSold").alias("item"))
    .filter(F.col("item.EmployeeId") != 0)
    .groupBy(F.col("item.EmployeeId").alias("EmployeeId"))
    .agg(
        F.sum("item.SoldPrice").alias("total_sales")
    )
)

display(employee_sales.orderBy(F.desc("total_sales")).limit(10))

EmployeeId,total_sales
13306,140224.60000000303
143217,136186.25000000227
2241,135968.48000000286
113408,131056.68000000188
28658,128374.77000000191
110556,126187.81000000141
14488,125823.9400000014
110343,123881.24000000168
44909,119657.75000000147
21618,118048.00000000102


In [0]:
# 6.1 - Employee check activity (Employee EDA methodology)

employee_item_activity = (
    master
    .select(
        "LocationId", "BusinessDate", "CheckId",
        F.explode("ItemsSold").alias("item")
    )
    .filter(F.col("item.EmployeeId") != 0)
    .groupBy(F.col("item.EmployeeId").alias("EmployeeId"))
    .agg(
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("n_checks")
    )
)

display(
    employee_item_activity
    .orderBy(F.desc("n_checks"))
    .limit(10)
)

EmployeeId,n_checks
13306,3854
143217,3661
44909,3483
113408,3442
28658,3385
14488,3369
21618,3352
110343,3284
2241,3209
110556,3193


In [0]:
# 6.1 - Employee EDA Metrics

employee_validation = (
    employee_item_activity
    .join(employee_sales, "EmployeeId", "inner")
    .join(
        employee_discount_risk.select(
            "EmployeeId", "discount_events", "total_discount_amount"
        ),
        "EmployeeId", "left"
    )
    .join(
        employee_void_risk.select(
            "EmployeeId", "void_events", "total_void_amount"
        ),
        "EmployeeId", "left"
    )
    .fillna(0, subset=[
        "discount_events", "total_discount_amount",
        "void_events", "total_void_amount"
    ])
    .filter((F.col("n_checks") >= 50) & (F.col("total_sales") > 0))
    .withColumn(
        "discount_rate_sales",
        F.col("total_discount_amount") / F.col("total_sales")
    )
    .withColumn(
        "void_rate_sales",
        F.col("total_void_amount") / F.col("total_sales")
    )
    .withColumn(
        "discounts_per_100",
        F.col("discount_events") / F.col("n_checks") * 100
    )
    .withColumn(
        "voids_per_100",
        F.col("void_events") / F.col("n_checks") * 100
    )
)

print("Empleados analizados:", employee_validation.count())
display(employee_validation.limit(10))

Empleados analizados: 2786


EmployeeId,n_checks,total_sales,discount_events,total_discount_amount,void_events,total_void_amount,discount_rate_sales,void_rate_sales,discounts_per_100,voids_per_100
56615,772,26941.769999999862,152,1002.29,23,97.95,0.03720208434709394,0.00363561859521481,19.689119170984455,2.9792746113989637
158868,174,7042.740000000004,4,27.4,34,98.96,0.003890531242101793,0.014051349332788081,2.2988505747126435,19.54022988505747
119787,516,23643.159999999833,77,393.03,31,149.16,0.0166234124372547,0.00630880136157777,14.922480620155037,6.007751937984496
158106,1512,60832.75999999941,0,0.0,149,749.1,0.0,0.0123140886588083,0.0,9.854497354497354
133107,1770,64790.28999999953,185,1588.85,148,642.93,0.024522964783766386,0.009923246214826398,10.451977401129943,8.361581920903955
153992,394,14067.039999999994,48,510.7,0,0.0,0.03630472366610177,0.0,12.18274111675127,0.0
157105,1102,40352.979999999734,47,417.9,11,50.37,0.010356112485372895,0.0012482349506777524,4.26497277676951,0.9981851179673321
158375,773,27899.909999999752,64,358.89,89,429.25,0.01286348235531954,0.015385354289673473,8.279430789133247,11.51358344113842
105640,1644,68917.39999999946,204,1057.83,63,313.84,0.015349244167655894,0.004553857226186746,12.408759124087592,3.832116788321168
142329,495,19833.06999999995,22,198.01,0,0.0,0.009983830037407245,0.0,4.444444444444445,0.0


In [0]:
# 6.1 - Reproduce Employee EDA anomaly flags

import pandas as pd

emp_validation_pd = employee_validation.toPandas()

def robust_z_active(s):
    s = s.astype(float)
    active = s[s > 0]

    if len(active) < 10:
        return pd.Series(0.0, index=s.index)

    med = active.median()
    mad = (active - med).abs().median()

    if mad == 0:
        return pd.Series(0.0, index=s.index)

    z = 0.6745 * (s - med) / mad
    return z.where(s > 0, 0.0)

metrics = [
    "discount_rate_sales",
    "discounts_per_100",
    "void_rate_sales",
    "voids_per_100"
]

for metric in metrics:
    emp_validation_pd[f"z_{metric}"] = robust_z_active(
        emp_validation_pd[metric]
    )
    emp_validation_pd[f"flag_{metric}"] = (
        emp_validation_pd[f"z_{metric}"] > 3.5
    )

emp_validation_pd["n_flags"] = emp_validation_pd[
    [f"flag_{m}" for m in metrics]
].sum(axis=1)

emp_validation_pd["risk_score"] = sum(
    emp_validation_pd[f"z_{m}"].clip(lower=0)
    for m in metrics
)

emp_validation_pd["atipico_fuerte"] = (
    emp_validation_pd["n_flags"] >= 2
)

print("Empleados analizados:", len(emp_validation_pd))
print("Atípicos fuertes:", emp_validation_pd["atipico_fuerte"].sum())

Empleados analizados: 2786
Atípicos fuertes: 145


In [0]:
# 6.1 - Cross-validation with Employee EDA

priority_ids = [
    int(row.EmployeeId)
    for row in priority_employees.select("EmployeeId").collect()
]

employee_cross_validation = (
    emp_validation_pd[
        emp_validation_pd["EmployeeId"].isin(priority_ids)
    ][
        [
            "EmployeeId",
            "n_checks",
            "n_flags",
            "risk_score",
            "atipico_fuerte"
        ]
    ]
    .sort_values(
        ["atipico_fuerte", "risk_score"],
        ascending=[False, False]
    )
)

display(employee_cross_validation)

print(
    "Empleados prioritarios identificados:",
    len(employee_cross_validation)
)

print(
    "Coincidencias con atípicos fuertes:",
    int(employee_cross_validation["atipico_fuerte"].sum())
)

EmployeeId,n_checks,n_flags,risk_score,atipico_fuerte
104918,253,4,28.037511561682702,true
161207,106,3,21.496036719479164,true
160872,182,3,20.459772661274044,true
161866,101,3,18.88024288811699,true
161089,142,4,18.172354244628103,true
114418,912,4,17.236971819326037,true
159766,187,2,15.215442882136518,true
161101,282,1,12.950305664760673,false
140589,112,1,12.747096356436629,false
154130,779,1,11.187311808178059,false


Empleados prioritarios identificados: 10
Coincidencias con atípicos fuertes: 7


#### Cross-Validation Insights

La comparación entre Fraud Pattern EDA y Employee EDA identificó una coincidencia de **7 de 10 empleados prioritarios (70%)** clasificados como atípicos fuertes bajo ambas metodologías.

Destacan los EmployeeId `104918`, `161089` y `114418`, que presentan las cuatro banderas de anomalía en Employee EDA. El empleado `104918` registra el mayor risk score del grupo (28.04).

Los tres empleados restantes presentan una bandera de anomalía, por lo que mantienen señales relevantes, aunque no cumplen el criterio de atípico fuerte.

Estos resultados refuerzan la priorización de ciertos empleados para una investigación operativa. Sin embargo, no demuestran fraude y deben interpretarse considerando que ambas metodologías comparten indicadores de descuentos y voids.

In [0]:
# 6.2 - Locations associated with priority employees

priority_locations = (
    master
    .select(
        "LocationId",
        "BusinessDate",
        "CheckId",
        F.explode("ItemsSold").alias("item")
    )
    .filter(
        F.col("item.EmployeeId").isin(
            [int(row.EmployeeId)
             for row in priority_employees.select("EmployeeId").collect()]
        )
    )
    .groupBy(
        F.col("item.EmployeeId").alias("EmployeeId"),
        "LocationId"
    )
    .agg(
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("checks_at_location")
    )
    .orderBy("EmployeeId", F.desc("checks_at_location"))
)

display(priority_locations)

EmployeeId,LocationId,checks_at_location
104918,196,253
114418,148,912
140589,2,112
154130,197,779
159766,95,187
160872,19,182
161089,54,142
161101,111,282
161207,197,106
161866,80,101


In [0]:
# 6.2 - Check access to Location EDA results

try:
    location_outliers = spark.table(
        "workspace.default.applebees_outliers"
    )

    print("Acceso disponible")
    print(location_outliers.columns)

except Exception as e:
    print("No se pudo acceder a la tabla.")
    print(str(e)[:350])

Acceso disponible
No se pudo acceder a la tabla.
[INSUFFICIENT_PERMISSIONS] Insufficient privileges:
User does not have SELECT on Table 'workspace.default.applebees_outliers'. SQLSTATE: 42501

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.sql.managedcatalog.client.TypeConversionUtils.toAnalysisException(TypeConversionUtils.scala:3931)
	at com.databricks.sql.managedcata


In [0]:
# 6.2 - Inspect Location EDA source columns

print([
    c for c in master.columns
    if any(
        word in c.lower()
        for word in ["location", "revenue", "discount", "void", "payment"]
    )
])

['LocationId', 'RevenueCenterId', 'Discounts', 'Payments', 'Voids', 'flag_no_payments', 'LocationName', 'RevenueCenterName', 'discount_count', 'discount_amount', 'max_discount', 'discount_employee_count', 'payment_count', 'payment_amount', 'payment_employee_count', 'void_count', 'void_amount', 'max_void', 'void_employee_count']


In [0]:
# 6.2 - Reproduce Location EDA features

from pyspark.sql import functions as F

def num(c):
    return F.coalesce(
        F.expr(f"try_cast({c} AS DOUBLE)"),
        F.lit(0.0)
    )

def flag(c):
    return (
        F.upper(F.col(c).cast("string")) == "TRUE"
    ).cast("int")

dur = (
    F.col("CloseTime").cast("long")
    - F.col("OpenTime").cast("long")
) / 60

location_base = (
    master
    .withColumn("sales", num("total_sold_price"))
    .withColumn("tip", F.abs(num("gratuity_raw")))
    .withColumn("disc", num("discount_amount"))
    .withColumn("voidamt", num("void_amount"))
    .withColumn("paid", (num("payment_amount") > 0).cast("int"))
    .withColumn("dur_min", dur)
)

location_features = (
    location_base
    .groupBy("LocationId", "LocationName", "RevenueCenterName")
    .agg(
        F.countDistinct("CheckId").alias("num_checks"),
        F.sum("sales").alias("sum_sales"),
        F.sum("disc").alias("sum_disc"),
        F.sum("voidamt").alias("sum_void"),
        F.sum(F.col("tip") * F.col("paid")).alias("sum_tip_paid"),
        F.sum(F.col("sales") * F.col("paid")).alias("sum_sales_paid"),
        F.avg(F.when(F.col("sales") > 0, F.col("sales"))).alias("avg_check"),
        F.avg(F.when(F.col("sales") > 0, num("total_item_quantity"))).alias("items_per_check"),
        F.avg((num("void_count") > 0).cast("int")).alias("void_rate"),
        F.avg(flag("flag_no_payments")).alias("comp_rate"),
        F.avg(flag("flag_empty_check")).alias("empty_rate"),
        F.avg(flag("flag_zero_duration")).alias("zero_dur_rate"),
        F.avg(
            F.when(F.col("dur_min").between(0, 360), F.col("dur_min"))
        ).alias("avg_duration_min")
    )
    .withColumn(
        "discount_pct",
        F.expr("try_divide(sum_disc, sum_sales + sum_disc)")
    )
    .withColumn(
        "void_pct",
        F.expr("try_divide(sum_void, sum_sales)")
    )
    .withColumn(
        "tip_pct",
        F.expr("try_divide(sum_tip_paid, sum_sales_paid)")
    )
    .filter(F.col("num_checks") >= 115)
)

print("Combinaciones analizadas:", location_features.count())

display(
    location_features
    .select(
        "LocationId",
        "RevenueCenterName",
        "num_checks",
        "discount_pct",
        "void_rate",
        "void_pct"
    )
    .limit(10)
)

Combinaciones analizadas: 629


LocationId,RevenueCenterName,num_checks,discount_pct,void_rate,void_pct
63,Online,969,0.007407205738608812,0.0020639834881320948,0.0012493297626176368
160,Uber Eats,342,0.0,0.0,0.0
144,Door Dash,1364,0.0,0.0,0.0
112,Online,1569,0.017419481922716668,0.0,0.0
3,To Go,435,0.09143302581967858,0.029885057471264367,0.028120755338269517
36,To Go,285,0.03828697643035507,0.017543859649122806,0.00832024082224967
165,Uber Eats,120,0.0,0.0,0.0
122,Uber Eats,1047,0.0,0.0,0.0
15,Uber Eats,122,0.0,0.0,0.0
179,To Go,467,0.020834397616006802,0.01284796573875803,0.008220800037100478


In [0]:
# 6.2 - Validate Location EDA aggregation

location_features = (
    location_features
    .withColumnRenamed("num_checks", "num_checks_previous")
)

location_check_counts = (
    location_base
    .groupBy(
        "LocationId",
        "LocationName",
        "RevenueCenterName"
    )
    .agg(
        F.count("*").alias("num_checks_corrected")
    )
)

location_features = (
    location_features
    .join(
        location_check_counts,
        ["LocationId", "LocationName", "RevenueCenterName"],
        "inner"
    )
)

print("Combinaciones analizadas:", location_features.count())

display(
    location_features
    .filter(
        F.col("num_checks_previous") !=
        F.col("num_checks_corrected")
    )
    .select(
        "LocationId",
        "RevenueCenterName",
        "num_checks_previous",
        "num_checks_corrected"
    )
    .limit(10)
)

Combinaciones analizadas: 629


LocationId,RevenueCenterName,num_checks_previous,num_checks_corrected
87,Olo - Mastercard,746,747
141,Door Dash,727,728
87,To Go,264,942
87,In-Store,423,22938
1,In-Store,342,21148
1,To Go,229,814
1,Door Dash,2472,2487
87,Door Dash,2539,2556
87,Olo - Visa,1123,1126
1,Olo - Visa,1179,1180


In [0]:
# 6.2 - Correct Location EDA aggregation

location_features = (
    location_base
    .groupBy(
        "LocationId",
        "RevenueCenterId",
        "LocationName",
        "RevenueCenterName"
    )
    .agg(
        F.count("*").alias("num_checks"),
        F.sum("sales").alias("sum_sales"),
        F.sum("disc").alias("sum_disc"),
        F.sum("voidamt").alias("sum_void"),
        F.sum(F.col("tip") * F.col("paid")).alias("sum_tip_paid"),
        F.sum(F.col("sales") * F.col("paid")).alias("sum_sales_paid"),
        F.avg(F.when(F.col("sales") > 0, F.col("sales"))).alias("avg_check"),
        F.avg(F.when(F.col("sales") > 0, num("total_item_quantity"))).alias("items_per_check"),
        F.avg((num("void_count") > 0).cast("int")).alias("void_rate"),
        F.avg(flag("flag_no_payments")).alias("comp_rate"),
        F.avg(flag("flag_empty_check")).alias("empty_rate"),
        F.avg(flag("flag_zero_duration")).alias("zero_dur_rate"),
        F.avg(
            F.when(F.col("dur_min").between(0, 360), F.col("dur_min"))
        ).alias("avg_duration_min")
    )
    .withColumn(
        "discount_pct",
        F.expr("try_divide(sum_disc, sum_sales + sum_disc)")
    )
    .withColumn(
        "void_pct",
        F.expr("try_divide(sum_void, sum_sales)")
    )
    .withColumn(
        "tip_pct",
        F.expr("try_divide(sum_tip_paid, sum_sales_paid)")
    )
    .filter(F.col("num_checks") >= 115)
)

print("Combinaciones válidas:", location_features.count())

display(
    location_features
    .orderBy(F.desc("num_checks"))
    .select(
        "LocationId",
        "RevenueCenterName",
        "num_checks",
        "discount_pct",
        "void_rate",
        "void_pct"
    )
    .limit(10)
)

Combinaciones válidas: 630


LocationId,RevenueCenterName,num_checks,discount_pct,void_rate,void_pct
87,In-Store,22938,0.0019086955531538196,0.030037492370738512,0.009604349963090605
198,In-Store,21439,0.022820278714558894,0.03521619478520453,0.010413977447105196
1,In-Store,21148,6.858866123686934E-4,0.03343105731038396,0.013018081086370554
170,In-Store,20694,0.01992391370573343,0.021842079829902387,0.005955678237077879
9,In-Store,19784,0.015792051127974164,0.02466639708855641,0.006750111133656165
5,In-Store,19594,0.020902724243503354,0.03587833010105134,0.009955811319589902
137,In-Store,19480,0.02401554907367417,0.03316221765913758,0.0082470814026912
121,In-Store,18533,0.017675410729892642,0.03253655641288512,0.009088808597505075
82,In-Store,18208,0.033602945095323265,0.037401142355008785,0.010208793486120582
166,In-Store,18106,0.01562427155493183,0.01872307522368276,0.004760195938477209


In [0]:
# 6.2 - Robust Z-Scores by Revenue Center

import numpy as np
import pandas as pd

location_pd = location_features.toPandas()

location_metrics = [
    "avg_check", "items_per_check", "discount_pct",
    "tip_pct", "void_rate", "void_pct",
    "comp_rate", "empty_rate",
    "zero_dur_rate", "avg_duration_min"
]

def robust_z_location(s):
    s = s.astype(float)
    median = s.median()
    deviation = (s - median).abs()
    mad = deviation.median()

    if mad > 0:
        return 0.6745 * (s - median) / mad

    mean_ad = deviation.mean()
    if mean_ad > 0:
        return (s - median) / (1.253314 * mean_ad)

    return s * 0.0

z_cols = []

for metric in location_metrics:
    z_col = f"{metric}_rz"

    location_pd[z_col] = (
        location_pd
        .groupby("RevenueCenterName")[metric]
        .transform(robust_z_location)
    )

    z_cols.append(z_col)

location_pd[z_cols] = location_pd[z_cols].fillna(0)

location_pd["max_abs_z"] = location_pd[z_cols].abs().max(axis=1)

location_pd["top_driver"] = (
    location_pd[z_cols]
    .abs()
    .idxmax(axis=1)
    .str.replace("_rz", "", regex=False)
)

location_pd["is_outlier_z"] = location_pd["max_abs_z"] > 3.5

print("Combinaciones analizadas:", len(location_pd))
print("Atípicos por z-score:", int(location_pd["is_outlier_z"].sum()))

Combinaciones analizadas: 630
Atípicos por z-score: 72


#### Robust Z-Score Results

Se analizaron 630 combinaciones de restaurante y revenue center mediante diez indicadores operativos.

El análisis identificó 72 combinaciones atípicas (11.43%), utilizando un umbral de z-score robusto absoluto superior a 3.5.

Estos resultados permiten identificar comportamientos inusuales respecto a ubicaciones del mismo canal, pero no constituyen evidencia directa de fraude.

In [0]:
# 6.2 - Isolation Forest

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

X = location_pd[location_metrics].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

X_scaled = RobustScaler().fit_transform(X)

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

location_pd["is_outlier_if"] = (
    iso.fit_predict(X_scaled) == -1
)

location_pd["is_outlier"] = (
    location_pd["is_outlier_z"] |
    location_pd["is_outlier_if"]
)

print("Atípicos por z-score:", int(location_pd["is_outlier_z"].sum()))
print("Atípicos por Isolation Forest:", int(location_pd["is_outlier_if"].sum()))
print("Atípicos por al menos un método:", int(location_pd["is_outlier"].sum()))

Atípicos por z-score: 72
Atípicos por Isolation Forest: 32
Atípicos por al menos un método: 79


#### Location Anomaly Detection Results

Se evaluaron 630 combinaciones de restaurante y revenue center utilizando Robust Z-Scores e Isolation Forest.

Se identificaron 72 combinaciones atípicas mediante Robust Z-Scores y 32 mediante Isolation Forest. En conjunto, 79 combinaciones (12.54%) fueron señaladas por al menos uno de los métodos, mientras que 25 coincidieron en ambos.

La combinación de métodos permite identificar distintos tipos de comportamiento inusual y priorizar ubicaciones para una revisión operativa, sin asumir que las anomalías representan fraude.

In [0]:
# 6.2 - Employee and Location Cross-Validation

priority_employee_ids = [
    114418, 154130, 161101, 104918, 159766,
    160872, 161089, 140589, 161207, 161866
]

priority_locations = (
    master
    .select(
        "LocationId",
        "BusinessDate",
        "CheckId",
        F.explode("ItemsSold").alias("item")
    )
    .filter(
        F.col("item.EmployeeId").isin(priority_employee_ids)
    )
    .groupBy(
        F.col("item.EmployeeId").alias("EmployeeId"),
        "LocationId"
    )
    .agg(
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("checks_at_location")
    )
    .orderBy("EmployeeId", F.desc("checks_at_location"))
)

priority_loc_pd = priority_locations.toPandas()

location_summary = (
    location_pd
    .groupby("LocationId")
    .agg(
        revenue_centers=("RevenueCenterName", "nunique"),
        anomalous_centers=("is_outlier", "sum"),
        max_abs_z=("max_abs_z", "max")
    )
    .reset_index()
)

location_summary["location_has_anomaly"] = (
    location_summary["anomalous_centers"] > 0
)

employee_location_cross = priority_loc_pd.merge(
    location_summary,
    on="LocationId",
    how="left"
)

display(
    employee_location_cross[
        [
            "EmployeeId",
            "LocationId",
            "checks_at_location",
            "revenue_centers",
            "anomalous_centers",
            "location_has_anomaly"
        ]
    ].sort_values(
        ["location_has_anomaly", "checks_at_location"],
        ascending=[False, False]
    )
)

EmployeeId,LocationId,checks_at_location,revenue_centers,anomalous_centers,location_has_anomaly
114418,148,912,5,2,true
159766,95,187,5,1,true
160872,19,182,5,1,true
161089,54,142,5,4,true
161866,80,101,6,1,true
154130,197,779,6,0,false
161101,111,282,4,0,false
104918,196,253,4,0,false
140589,2,112,5,0,false
161207,197,106,6,0,false


#### Location-Level Cross-Validation Results

Se cruzaron los diez empleados prioritarios del análisis de fraude con los resultados de detección de anomalías por ubicación.

Cinco empleados (50%) están asociados con restaurantes que presentan al menos un revenue center atípico. Destaca el empleado 161089, cuya ubicación (LocationId 54) presenta anomalías en cuatro de cinco canales analizados.

Los cinco empleados también fueron clasificados como atípicos fuertes en Employee EDA, lo que refuerza su prioridad para una revisión operativa.

Sin embargo, la coincidencia se identifica a nivel restaurante y no demuestra que las transacciones de cada empleado hayan ocurrido en los revenue centers atípicos. Los resultados tampoco constituyen evidencia concluyente de fraude.

In [0]:
# 6.2 - Employee and Revenue Center Cross-Validation

employee_center_activity = (
    master
    .select(
        "LocationId",
        "RevenueCenterId",
        "BusinessDate",
        "CheckId",
        F.explode("ItemsSold").alias("item")
    )
    .filter(F.col("item.EmployeeId").isin(priority_employee_ids))
    .groupBy(
        F.col("item.EmployeeId").alias("EmployeeId"),
        "LocationId",
        "RevenueCenterId"
    )
    .agg(
        F.countDistinct(
            "LocationId", "BusinessDate", "CheckId"
        ).alias("checks_at_center")
    )
    .toPandas()
)

center_flags = location_pd[
    ["LocationId", "RevenueCenterId", "RevenueCenterName", "is_outlier"]
]

employee_center_cross = employee_center_activity.merge(
    center_flags,
    on=["LocationId", "RevenueCenterId"],
    how="left"
)

display(
    employee_center_cross[
        employee_center_cross["is_outlier"] == True
    ].sort_values(
        "checks_at_center", ascending=False
    )
)

EmployeeId,LocationId,RevenueCenterId,checks_at_center,RevenueCenterName,is_outlier
114418,148,1,902,In-Store,true
161089,54,1,141,In-Store,true
161866,80,1,101,In-Store,true
160872,19,4,8,Online,true
114418,148,4,1,Online,true


#### Location-Level Cross-Validation: Final Findings

Se cruzaron los diez empleados prioritarios con las anomalías identificadas mediante Robust Z-Scores e Isolation Forest a nivel de restaurante y revenue center.

Inicialmente, cinco empleados (50%) estaban asociados con restaurantes que presentaban al menos un canal atípico. Sin embargo, al validar los revenue centers específicos donde registraron actividad, la coincidencia se redujo a cuatro empleados (40%).

Destacan los empleados 114418, 161089 y 161866, con 902, 141 y 101 checks respectivamente en canales In-Store clasificados como atípicos. El empleado 160872 también presentó una coincidencia, con ocho checks en Online.

Estos resultados permiten priorizar revisiones a nivel empleado, ubicación y canal. Sin embargo, la presencia de actividad en un revenue center atípico no demuestra que las transacciones individuales sean fraudulentas.

In [0]:
# 6.3 - Transaction & Time Cross-Validation (Fixed)

from pyspark.sql import functions as F

priority_ids = [
    114418, 154130, 161101, 104918, 159766,
    160872, 161089, 140589, 161207, 161866
]

time_cross = (
    master
    .select(
        "LocationId", "BusinessDate", "CheckId",
        "OpenTime", "discount_amount", "void_amount",
        F.explode("ItemsSold").alias("item")
    )
    .withColumn("EmployeeId", F.col("item.EmployeeId"))
    .filter(F.col("EmployeeId").isin(priority_ids))
    .dropDuplicates([
        "LocationId", "BusinessDate", "CheckId", "EmployeeId"
    ])
    .withColumn("hour", F.hour("OpenTime"))
    .groupBy("EmployeeId")
    .agg(
        F.count("*").alias("checks"),
        F.sum((F.col("discount_amount") > 0).cast("int")).alias("discount_checks"),
        F.sum((F.col("void_amount") > 0).cast("int")).alias("void_checks"),
        F.sum(
            F.when(
                (F.col("hour") < 10) | (F.col("hour") >= 22), 1
            ).otherwise(0)
        ).alias("checks_outside_10_22")
    )
    .withColumn(
        "outside_hours_pct",
        F.round(
            100 * F.col("checks_outside_10_22") / F.col("checks"), 2
        )
    )
    .orderBy(F.desc("outside_hours_pct"))
)

display(time_cross)

EmployeeId,checks,discount_checks,void_checks,checks_outside_10_22,outside_hours_pct
104918,253,79,22,59,23.32
160872,182,42,11,17,9.34
161101,282,56,21,18,6.38
161089,142,27,12,9,6.34
161866,101,20,8,3,2.97
161207,106,21,10,3,2.83
114418,912,179,97,10,1.1
159766,187,35,18,2,1.07
154130,779,165,63,0,0.0
140589,112,29,11,0,0.0


#### Transaction & Time Cross-Validation Results

Se analizaron los horarios de apertura de checks asociados con los diez empleados prioritarios, utilizando el intervalo exploratorio de 10:00 a 22:00.

El empleado 104918 presentó la mayor proporción de checks fuera de este intervalo (23.32%, 59 de 253), seguido de 160872 (9.34%) y 161101 (6.38%).

El empleado 104918 también fue identificado como atípico fuerte en Employee EDA, por lo que constituye un caso relevante para revisión.

Estos resultados no confirman actividad fraudulenta, ya que el intervalo utilizado no representa necesariamente los horarios oficiales de operación. Asimismo, los descuentos y voids corresponden al check completo y no pueden atribuirse automáticamente al empleado.

#### Cross-Validation Summary

El análisis cruzado permitió identificar coincidencias entre patrones de descuentos y voids, comportamiento atípico de empleados, anomalías por ubicación y horarios de transacciones.

De los diez empleados prioritarios, siete fueron clasificados como atípicos fuertes en Employee EDA y cuatro registraron actividad en revenue centers atípicos. Adicionalmente, el empleado 104918 presentó la mayor proporción de checks fuera del intervalo horario exploratorio (23.32%).

Los resultados permiten priorizar casos para una revisión operativa más detallada, pero no constituyen evidencia concluyente de fraude. Las coincidencias entre análisis deben interpretarse considerando que comparten información de origen.